<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [1]:
# Install spark
!pip install pyspark
!pip install findspark

In [2]:
# Start session
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("e-commerce data analysis").getOrCreate()

26/05/20 14:46:14 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 14:46:16 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Download The search term dataset from the below url
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

--2026-05-20 14:46:22--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv.1’

searchterms.csv.1   100%[===================>] 227.99K  --.-KB/s    in 0.03s   

2026-05-20 14:46:22 (6.48 MB/s) - ‘searchterms.csv.1’ saved [233457/233457]



In [4]:
# Load the csv into a spark dataframe
df = spark.read.csv("searchterms.csv", header=True, inferSchema=True)

In [5]:
# Print the number of rows and columns
print((df.count(), len(df.columns)))

(10000, 4)


In [6]:
# Print the top 5 rows
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



In [7]:
# Find out the datatype of the column searchterm?
df.printSchema()

root
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- searchterm: string (nullable = true)



In [8]:
# How many times was the term `gaming laptop` searched?
df.filter("searchterm == 'gaming laptop'").count()

499

In [9]:
# Print the top 5 most frequently used search terms?
df.groupby("searchterm").count().orderBy("count", ascending=False).show(5)

[Stage 8:===============================================>      (175 + 13) / 200]

+-------------+-----+
|   searchterm|count|
+-------------+-----+
|mobile 6 inch| 2312|
|    mobile 5g| 2301|
|mobile latest| 1327|
|       laptop|  935|
|  tablet wifi|  896|
+-------------+-----+
only showing top 5 rows



In [10]:
# The pretrained sales forecasting model is available at  the below url
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

--2026-05-20 14:46:47--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1490 (1.5K) [application/x-tar]
Saving to: ‘model.tar.gz.1’

model.tar.gz.1      100%[===================>]   1.46K  --.-KB/s    in 0s      

2026-05-20 14:46:47 (12.6 MB/s) - ‘model.tar.gz.1’ saved [1490/1490]



In [11]:
# Load the sales forecast model.
import tarfile

with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall(path="./model")

In [12]:
!ls model

sales_prediction.model


In [13]:
from pyspark.ml.regression import LinearRegressionModel

model = LinearRegressionModel.load("./model/sales_prediction.model")

In [14]:
# Using the sales forecast model, predict the sales for the year of 2023.
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["year"], outputCol="features")
data = [[2023, 0]]
columns = ["year", "sales"]
df = spark.createDataFrame(data, columns)
transformed_df = assembler.transform(df).select("features", "sales")
predictions = model.transform(transformed_df)
predictions.select("prediction").show()

+------------------+
|        prediction|
+------------------+
|175.16564294006457|
+------------------+



26/05/20 14:46:56 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/05/20 14:46:56 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
